# ДЗ 2. Функции и словари на настоящих данных
Срок: **до понедельника 19:00**.

**Сдача через GitHub** (инструкция «Git для домашних заданий»): сохраните ноутбук как `hw02.ipynb` в папку `hw02` репозитория `dsa-homework`, затем `git add hw02` → `git commit -m "hw02: ..."` → `git push`. Из Colab: Файл → Скачать → Скачать .ipynb. Файл `orders.csv` в репозиторий класть не нужно.

Файл `orders.csv` — заказы «Дала Маркет» за сентябрь и октябрь 2026. Работаем **без Pandas и NumPy**: только `csv`, функции, словари, множества.

- **[База]** обязательно для всех.
- **[Стандарт]** решайте, сколько получится.
- **[Челлендж]** для тех, кому легко.

**Правило AI:** можно спросить, *почему* не работает код, но нельзя просить решение.

## [Для всех] Чтение кода — без запуска
Для каждого фрагмента: **сначала** напишите в Markdown-ячейке ответ и одно предложение «почему», **потом** запустите и сравните. Ошибка в предсказании не снижает оценку — снижает отсутствие объяснения.
```python
# 1
r = {}
for c in ["A", "B", "A"]:
    r[c] = r.get(c, 0) + 1
print(r)

# 2
def f(x):
    x.append(1)
a = [0]; f(a); f(a); print(a)

# 3
s = {c.lower() for c in ["Almaty", "ALMATY", "Astana"]}; print(len(s))

# 4
d = {k: len(k) for k in ["ab", "abc", "ab"]}; print(d)

# 5
def g(n):
    if n > 0:
        print(n)
x = g(5); print(x)
```
На следующем занятии несколько человек объяснят вслух одну строку своего ДЗ.

Мои предсказания и объяснения:

1. {'A': 2, 'B': 1}
2. [0,1,1]
3. 3
4. {'ab': 2, 'abc': 3, 'ab': 2}
5. 5, None

In [23]:
import csv
with open("orders.csv", encoding="utf-8") as f:
    rows = list(csv.DictReader(f))
print(len(rows), "строк")

5463 строк


In [24]:
# Загрузка orders.csv: в Colab нажмите «Выбрать файлы» и укажите orders.csv из LMS.
# Если файл уже лежит рядом с ноутбуком, загрузка пропускается.
import os
if not os.path.exists("orders.csv"):
    try:
        from google.colab import files
        files.upload()
    except ImportError:
        print("Положите orders.csv в одну папку с ноутбуком")
print("orders.csv на месте:", os.path.exists("orders.csv"))

orders.csv на месте: True


## [База]
1. Напишите функции `to_num(s)` и `clean_city(s)` (6 городов, латиница и разный регистр → кириллица с заглавной).
2. Сколько заказов в каждой категории (`category`)? Отсортируйте по убыванию.
3. Для каждой категории посчитайте выручку оплаченных заказов и средний чек. Выведите таблицу f-строкой.
4. Сколько заказов в каждом месяце? Месяц — это `order_date[:7]`.

In [25]:
# База
#1
def to_num(s):
    s = s.replace(" ", "").replace("\xa0", "").replace(",", ".")
    return float(s) if s else None

CITIES = {"алматы": "Алматы", "almaty": "Алматы", "астана": "Астана", "astana": "Астана",
          "шымкент": "Шымкент", "shymkent": "Шымкент", "караганда": "Караганда",
          "актобе": "Актобе", "атырау": "Атырау"}

def clean_city(s):
    return CITIES[s.strip().lower()]

for r in rows:
    r["amount"] = to_num(r["amount"])
    r["city"] = clean_city(r["city"])

paid = [r for r in rows if r["status"] == "оплачен"]
#2
cnt = {}
for r in rows:
    cnt[r["category"]] = cnt.get(r["category"], 0) + 1
for k, v in sorted(cnt.items(), key=lambda x: -x[1]):
    print(f"{k:<12}{v:>6}")
#3
rev, n = {}, {}
for r in paid:
    c = r["category"]
    rev[c] = rev.get(c, 0) + r["amount"]
    n[c] = n.get(c, 0) + 1
print(f"{'Категория':<12}{'Выручка':>14}{'Ср. чек':>12}")
for c in sorted(rev, key=rev.get, reverse=True):
    print(f"{c:<12}{rev[c]:>14,.0f}{rev[c]/n[c]:>12,.0f}")
#4
by_m = {}
for r in rows:
    m = r["order_date"][:7]
    by_m[m] = by_m.get(m, 0) + 1
print(by_m)



Одежда        1529
Электроника   1185
Дом           1105
Красота        991
Книги          653
Категория          Выручка     Ср. чек
Электроника     65,270,240      61,287
Одежда          18,321,080      13,286
Дом             10,516,320      10,655
Красота          5,507,620       6,188
Книги            2,294,100       3,935
{'2026-09': 2782, '2026-10': 2681}


## [Стандарт]
5. Выручка оплаченных по месяцам и изменение октября к сентябрю в процентах.
6. Доля отменённых заказов по каждому (чистому) городу, в процентах с одним знаком.
7. Топ-5 клиентов (`customer_id`) по выручке оплаченных заказов. Кто на первом месте и что с ним не так? Ответьте в Markdown-ячейке.
8. Сколько **уникальных** клиентов в каждом городе? Подсказка: словарь множеств.

In [26]:
# Стандарт
#5
def monthly(data):
    d = {}
    for r in data:
        m = r["order_date"][:7]
        d[m] = d.get(m, 0) + r["amount"]
    return d

def change(d):
    return (d["2026-10"] / d["2026-09"] - 1) * 100

m5 = monthly(paid)
print(m5, f"{change(m5):+.1f}%")
#6
tot, canc = {}, {}
for r in rows:
    c = r["city"]
    tot[c] = tot.get(c, 0) + 1
    if r["status"] == "отменён":
        canc[c] = canc.get(c, 0) + 1
for c in sorted(tot, key=lambda c: -canc.get(c, 0) / tot[c]):
    print(f"{c:<12}{canc.get(c, 0) / tot[c] * 100:>6.1f}%")
#7
cl = {}
for r in paid:
    cl[r["customer_id"]] = cl.get(r["customer_id"], 0) + r["amount"]
top5 = sorted(cl.items(), key=lambda x: -x[1])[:5]
for c, v in top5:
    print(c, f"{v:,.0f}")
big = top5[0][0]
#8
u = {}
for r in rows:
    u.setdefault(r["city"], set()).add(r["customer_id"])
for c, s in sorted(u.items(), key=lambda x: -len(x[1])):
    print(c, len(s))


{'2026-09': 57489850.0, '2026-10': 44419510.0} -22.7%
Шымкент       20.4%
Актобе        10.3%
Караганда      8.9%
Астана         8.8%
Алматы         7.9%
Атырау         7.5%
TEST 29,999,970
C01627 257,180
C00485 215,920
C01338 192,080
C02122 191,670
Алматы 1351
Астана 981
Шымкент 655
Караганда 451
Актобе 358
Атырау 138


## [Челлендж]
9. Найдите полные дубли строк: превратите каждую строку в `tuple(r.values())` и проверяйте через множество. Сколько их и в каком месяце?
10. Пересчитайте выручку по месяцам без дублей и без клиента из задачи 7. Как изменился процент из задачи 5?
11. У какого клиента больше всего заказов (после удаления дублей)? Сколько?
12. Напишите менеджеру 3–4 предложения: насколько на самом деле упала выручка и почему первый расчёт был неверным.

In [27]:
# Челлендж
#9
seen, dups = set(), []
for r in rows:
    t = tuple(r.values())
    if t in seen:
        dups.append(r)
    else:
        seen.add(t)
dm = {}
for r in dups:
    m = r["order_date"][:7]
    dm[m] = dm.get(m, 0) + 1
print(len(dups), dm)
#10
seen, clean = set(), []
for r in paid:
    t = tuple(r.values())
    if t in seen or r["customer_id"] == big:
        continue
    seen.add(t)
    clean.append(r)
m10 = monthly(clean)
print(m10, f"{change(m10):+.1f}%")
print("Падение выручки в задаче 5 (−22,7%) было ложным. Его создали 3 тестовых заказа (два из них в сентябре) и 180 дублей, тоже сентябрьских. После очистки октябрь просел всего на 2,3%.")
#11
seen, nodup = set(), []
for r in rows:
    t = tuple(r.values())
    if t in seen:
        continue
    seen.add(t)
    nodup.append(r)

oc = {}
for r in nodup:
    oc[r["customer_id"]] = oc.get(r["customer_id"], 0) + 1

top = sorted(oc.items(), key=lambda x: -x[1])[:5]
print(top)

#12



180 {'2026-09': 180}
{'2026-09': 35243410.0, '2026-10': 34419520.0} -2.3%
Падение выручки в задаче 5 (−22,7%) было ложным. Его создали 3 тестовых заказа (два из них в сентябре) и 180 дублей, тоже сентябрьских. После очистки октябрь просел всего на 2,3%.
[('C01818', 10), ('C01076', 9), ('C01962', 8), ('C01947', 8), ('C02091', 8)]
